In [6]:
## dataset 

digits = load_digits()

In [35]:
data = data.reshape((1797, 64, )) ## reshaping the input befor passing the input to MinMaxScaler

In [37]:
## scaling the input

min_max_sc = MinMaxScaler()
X = min_max_sc.fit_transform(data)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.25, random_state=42)

---## Standardized ML Pipeline*Auto-generated by Phase 5 ML Standardization***STEP 1** — LazyPredict baseline comparison  **STEP 2** — PyCaret automated pipeline

### STEP 1 — LazyPredict: Baseline Model ComparisonRun all sklearn-compatible models to find the best baseline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from lazypredict.Supervised import LazyRegressor

# Use existing train/test split from preprocessing above
# Ensure numeric-only for LazyPredict (handles mixed types)
try:
    X_train_lp = X_train.select_dtypes(include=['number']).fillna(0)
    X_test_lp = X_test.select_dtypes(include=['number']).fillna(0)
except AttributeError:
    X_train_lp, X_test_lp = X_train, X_test

# Run LazyPredict
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions_df = reg.fit(X_train_lp, X_test_lp, y_train, y_test)

print("=" * 60)
print("LazyPredict — Regression Baseline Results")
print("=" * 60)
models_df

#### Top Models Visualization

In [ ]:
import matplotlib.pyplot as plt

top_n = min(15, len(models_df))
fig, ax = plt.subplots(figsize=(10, 6))
models_df['R-Squared'].head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('R-Squared')
ax.set_title(f'Top {top_n} Models — R-Squared')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### STEP 2 — PyCaret: Automated ML PipelineFull pipeline with automated preprocessing, model comparison, tuning, and finalization.> **Note:** PyCaret requires Python 3.9–3.11. Install with: `pip install pycaret`

In [ ]:
import sys
import pandas as pd

# PyCaret setup
try:
    from pycaret.regression import setup, compare_models, finalize_model, predict_model, save_model, plot_model
except ImportError:
    print("PyCaret not installed. Install with: pip install pycaret")
    print("Requires Python 3.9-3.11")
    raise SystemExit("PyCaret required for STEP 2")

# Safety check — target variable must exist
assert y_train is not None, "y_train is None — cannot proceed"

# Reconstruct DataFrame for PyCaret (needs column-named target)
# IMPORTANT: Do NOT mutate original X_train / y_train
if isinstance(X_train, pd.DataFrame):
    df_train = X_train.copy()
else:
    df_train = pd.DataFrame(X_train)
df_train['__target__'] = y_train

if isinstance(X_test, pd.DataFrame):
    df_test = X_test.copy()
else:
    df_test = pd.DataFrame(X_test)
df_test['__target__'] = y_test

_full_df = pd.concat([df_train, df_test], ignore_index=True)

# Configure PyCaret session
reg_setup = setup(
    data=_full_df,
    target='__target__',
    session_id=42,
    silent=True,
    verbose=False,
    fold=5,
)
print("PyCaret setup complete.")

In [ ]:
# Compare all models
best_model = compare_models(sort='R2', n_select=1)
print(f"\nBest model: {best_model}")

In [ ]:
# Finalize the best model (retrain on full dataset)
final_model = finalize_model(best_model)
print(f"Finalized model: {final_model}")

#### Model Evaluation

In [ ]:
# Model evaluation plots
try:
    plot_model(best_model, plot='residuals', save=True)
    plot_model(best_model, plot='error', save=True)
    plot_model(best_model, plot='feature', save=True)
except Exception as e:
    print(f"Plot generation note: {e}")

#### Save Model Pipeline

In [ ]:
# Save the final pipeline
save_model(final_model, 'final_model_pipeline')
print("Model saved as 'final_model_pipeline.pkl'")